# 03 — Exploratory Data Analysis & Strategic Insights

This notebook investigates core business hypotheses across sales trajectories, customer RFM behaviors, delivery delays, and CSAT scores.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import PROCESSED_DATA_DIR

# Load processed dimensional star schema
dim_date = pd.read_csv(PROCESSED_DATA_DIR / "dim_date.csv")
dim_customer = pd.read_csv(PROCESSED_DATA_DIR / "dim_customer.csv")
dim_product = pd.read_csv(PROCESSED_DATA_DIR / "dim_product.csv")
fact_orders = pd.read_csv(PROCESSED_DATA_DIR / "fact_orders.csv")
fact_sales = pd.read_csv(PROCESSED_DATA_DIR / "fact_sales.csv")
fact_reviews = pd.read_csv(PROCESSED_DATA_DIR / "fact_reviews.csv")


### 1. Monthly Revenue Trajectory & Seasonality


In [ ]:
monthly = fact_sales.merge(dim_date, left_on="date_key", right_on="date_key")
monthly_rev = monthly.groupby("year_month")["item_value"].sum().reset_index()
print(monthly_rev.tail(12))


### 2. Category Pareto & Revenue Concentration


In [ ]:
cat_rev = fact_sales.merge(dim_product, on="product_key").groupby("product_category_name_english").agg(
    total_revenue=("item_value", "sum"),
    units_sold=("sales_key", "count")
).reset_index().sort_values(by="total_revenue", ascending=False)

print(cat_rev.head(10))


### 3. Customer RFM Segmentation Breakdown


In [ ]:
rfm_dist = dim_customer.groupby("rfm_segment").agg(
    customer_count=("customer_unique_id", "nunique"),
    total_spend=("rfm_monetary", "sum"),
    avg_recency=("rfm_recency", "mean")
).reset_index().sort_values(by="total_spend", ascending=False)

print(rfm_dist)


### 4. Operational Latency vs Customer Review Score (CSAT)


In [ ]:
deliv_csat = fact_orders[fact_orders["order_status"] == "delivered"].merge(fact_reviews, on="order_id")
print(deliv_csat.groupby("is_late").agg(
    order_count=("order_id", "count"),
    avg_review_score=("review_score", "mean"),
    one_star_pct=("review_score", lambda x: (x == 1).mean() * 100),
    five_star_pct=("review_score", lambda x: (x == 5).mean() * 100)
))
